# BatteryLife 特征相关性与模型误差分析

本 Notebook 面向中文读者，说明本次分析如何联动当前 BatteryLife 仓库与 `SOHbenchmark_TSLib` 仓库，并复现以下三类问题：

1. 每个数据集、每个特征与 SOH/RUL 真值之间的相关性；
2. 每个特征与模型输出、绝对误差、相对误差之间的关系；
3. 特征相关性结构是否能映射到模型性能，尤其是“相关性越高，模型潜在误差是否越小”。


## 指标定义

- **Pearson 相关系数**：衡量两个变量的线性关系，取值范围 [-1, 1]。绝对值越大，线性关系越强。
- **Spearman 秩相关系数**：先把变量转换成秩，再计算 Pearson，衡量单调关系。它比 Pearson 更适合电池退化这种非线性但单调的过程。
- **|Spearman|**：忽略方向，只衡量相关强度。例如 -0.9 和 0.9 都表示很强的单调关系。
- **特征-误差敏感度**：对每个 `res.npz` 中的逐样本预测，计算某个特征与 `|y_pred - y_true|` 的 Spearman。其绝对值越大，说明误差更强地沿该特征方向变化。
- **相对误差敏感度**：把绝对误差换成 `|y_pred - y_true| / |y_true|` 后重复上述计算。
- **数据集层面的相关性画像**：对一个数据集的 16 个特征，计算平均、最大、Top-3 平均的 |Spearman|，再与模型的 MAE/MAPE/RMSE/R2/Pearson 进行二级相关分析。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
BASE = Path('outputs_tslib')
if not BASE.exists():
    BASE = Path('tutorials/correlation_analysis/outputs_tslib')
FIG = BASE / 'figures'
truth = pd.read_csv(BASE / 'feature_truth_correlations_by_dataset.csv')
pred_rel = pd.read_csv(BASE / 'feature_prediction_error_relationships_available_npz.csv')
feature_hyp = pd.read_csv(BASE / 'hypothesis_feature_error_sensitivity.csv')
dataset_hyp = pd.read_csv(BASE / 'hypothesis_dataset_model_performance.csv')
summary = pd.read_csv(BASE / 'hypothesis_test_summary.csv')
truth.head()

## 每个数据集、每个特征与真值的相关性热图

下面两张图分别对应 SOH 和 RUL。行是数据集，列是 16 个特征，颜色表示 `|Spearman|`。

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIG / 'truth_corr_heatmap_SOH.png')))
display(Image(filename=str(FIG / 'truth_corr_heatmap_RUL.png')))

## 遍历每个数据集：每个特征的相关性柱状图

下面的代码会按目标和数据集遍历，逐一显示 16 个特征与真值的 `|Spearman|`。如需保存，图片已经在 `outputs_tslib/figures/` 中生成。

In [ ]:
for target in ['SOH', 'RUL']:
    print(f'===== {target} =====')
    for dataset in sorted(truth['dataset'].unique()):
        path = FIG / f'truth_corr_bar_{target}_{dataset}.png'
        if path.exists():
            print(dataset)
            display(Image(filename=str(path)))

## 特征与模型误差的敏感度

下面热图显示：每个任务、每个特征与绝对误差的平均 `|Spearman|`。颜色越深，说明模型误差越沿该特征方向变化。

In [ ]:
display(Image(filename=str(FIG / 'error_sensitivity_heatmap_SOH.png')))
display(Image(filename=str(FIG / 'error_sensitivity_heatmap_RUL.png')))

## 核心假设检验：相关性越高，误差是否越小？

先看特征级别：横轴是特征与真值的 `|Spearman|`，纵轴是特征与绝对误差的平均 `|Spearman|`。如果“相关性越高，误差越小”在特征级别成立，我们应该看到负相关；如果是正相关，则说明高相关特征也可能是误差变化最敏感的方向。

In [ ]:
display(Image(filename=str(FIG / 'feature_corr_vs_error_sensitivity_SOH.png')))
display(Image(filename=str(FIG / 'feature_corr_vs_error_sensitivity_RUL.png')))
summary[summary['level'] == 'feature_error_sensitivity']

## 数据集/模型层面的假设检验

再看数据集层面：用一个数据集中最强特征-真值相关性，去解释不同模型在该任务上的平均 MAPE。这里更接近“这个数据集是否更容易预测”。

In [ ]:
display(Image(filename=str(FIG / 'dataset_max_corr_vs_mape_SOH.png')))
display(Image(filename=str(FIG / 'dataset_max_corr_vs_mape_RUL.png')))
summary[summary['level'] == 'dataset_model_performance'].sort_values('spearman', key=lambda s: s.abs(), ascending=False).head(20)

## 结论

从当前结果看，不能简单说“单个特征相关性越高，模型逐样本误差越小”。在特征级别，高相关特征反而往往也是误差最敏感的方向之一。但是，在数据集整体层面，若某个数据集存在很强的特征-真值单调关系，模型通常会表现出更好的相对误差或拟合质量。因此更准确的表述是：高相关性提高了数据集层面的可预测性，但不保证逐样本误差单调降低。